# Week 7: Movie Ratings

**Sebastian Perdomo | IS 362**

Six raters scored six 2025 movies on a 1 to 5 scale. Blank cells are movies a rater
hasn't seen. The ratings are sample data.

## 1. Load the ratings

In [1]:
import pandas as pd

ratings = pd.read_csv("movie_ratings.csv", index_col="Rater").astype("Int64")
ratings

,Sinners,Superman,Jurassic World Rebirth,F1,Weapons,Zootopia 2
Rater,,,,,,
Alex,5,5,4,5,4,5
Brianna,3,2,1,2,3,<NA>
Carlos,5,3,1,4,5,2
Dana,4,<NA>,3,<NA>,5,4
Evan,4,4,3,5,<NA>,3
Fatima,<NA>,4,2,3,4,5


## 2. Average ratings

pandas skips missing values when it averages, so each average only counts the movies
that person actually saw.

In [2]:
print("Average rating by rater:")
print(ratings.mean(axis=1).round(2))
print()
print("Average rating by movie:")
print(ratings.mean().round(2))

Average rating by rater:
Rater
Alex       4.67
Brianna     2.2
Carlos     3.33
Dana        4.0
Evan        3.8
Fatima      3.6
dtype: Float64

Average rating by movie:
Sinners                    4.2
Superman                   3.6
Jurassic World Rebirth    2.33
F1                         3.8
Weapons                    4.2
Zootopia 2                 3.8
dtype: Float64


## 3. Normalized ratings

Each rater's scores are rescaled from 0 to 1. Their lowest rating becomes 0 and their
highest becomes 1, so a harsh rater and a generous rater end up on the same scale.

In [3]:
low = ratings.min(axis=1)
high = ratings.max(axis=1)

normalized = ratings.sub(low, axis=0).div(high - low, axis=0)
normalized.round(2)

,Sinners,Superman,Jurassic World Rebirth,F1,Weapons,Zootopia 2
Rater,,,,,,
Alex,1.0,1.0,0.0,1.0,0.0,1.0
Brianna,1.0,0.5,0.0,0.5,1.0,<NA>
Carlos,1.0,0.5,0.0,0.75,1.0,0.25
Dana,0.5,<NA>,0.0,<NA>,1.0,0.5
Evan,0.5,0.5,0.0,1.0,<NA>,0.0
Fatima,<NA>,0.67,0.0,0.33,0.67,1.0


In [4]:
print("Average normalized rating by rater:")
print(normalized.mean(axis=1).round(2))
print()
print("Average normalized rating by movie:")
print(normalized.mean().round(2))

Average normalized rating by rater:
Rater
Alex       0.67
Brianna     0.6
Carlos     0.58
Dana        0.5
Evan        0.4
Fatima     0.53
dtype: Float64

Average normalized rating by movie:
Sinners                    0.8
Superman                  0.63
Jurassic World Rebirth     0.0
F1                        0.72
Weapons                   0.73
Zootopia 2                0.55
dtype: Float64


## 4. Normalized versus actual ratings

In [5]:
comparison = pd.DataFrame({
    "Actual Avg": ratings.mean().round(2),
    "Actual Rank": ratings.mean().rank(ascending=False, method="min").astype(int),
    "Normalized Avg": normalized.mean().round(2),
    "Normalized Rank": normalized.mean().rank(ascending=False, method="min").astype(int),
})
comparison.sort_values("Normalized Rank")

,Actual Avg,Actual Rank,Normalized Avg,Normalized Rank
Sinners,4.2,1,0.8,1
Weapons,4.2,1,0.73,2
F1,3.8,3,0.72,3
Superman,3.6,5,0.63,4
Zootopia 2,3.8,3,0.55,5
Jurassic World Rebirth,2.33,6,0.0,6


**Advantages.** Normalizing removes each rater's personal habits. Alex rates almost
everything a 5 and Brianna rarely goes above 3, so a 3 from Brianna means about as much
as a 5 from Alex. Normalizing also fixes a bias in which movies people saw. On actual
ratings, Zootopia 2 ties F1. But Brianna, the harshest rater, never saw Zootopia 2, so
her low scores never pulled its average down. Once everyone is on the same scale, F1
pulls ahead. Normalizing also breaks the tie between Sinners and Weapons.

**Disadvantages.** Normalizing throws away the actual score. Alex never rated anything
below 4, so his 4s become 0, the same value as Brianna's 1, even though he liked those
movies. The rater averages also stop meaning much. Evan's average drops to the lowest in
the group even though his actual ratings are above average. The method also depends on
each person's range, so a rater who only saw two movies gets stretched to the full 0 to
1 scale, and a rater who gives every movie the same score can't be normalized at all,
since that means dividing by zero.

## 5. Standardized ratings (extra credit)

Standardizing turns each rating into a z-score: how many standard deviations it sits
above or below that rater's own average. A positive score means the rater liked the
movie more than usual.

In [6]:
standardized = (ratings.sub(ratings.mean(axis=1), axis=0)
                       .div(ratings.std(axis=1), axis=0))
standardized.round(2)

,Sinners,Superman,Jurassic World Rebirth,F1,Weapons,Zootopia 2
Rater,,,,,,
Alex,0.65,0.65,-1.29,0.65,-1.29,0.65
Brianna,0.96,-0.24,-1.43,-0.24,0.96,<NA>
Carlos,1.02,-0.2,-1.43,0.41,1.02,-0.82
Dana,0.0,<NA>,-1.22,<NA>,1.22,0.0
Evan,0.24,0.24,-0.96,1.43,<NA>,-0.96
Fatima,<NA>,0.35,-1.4,-0.53,0.35,1.23


In [7]:
print("Average standardized rating by rater:")
print(standardized.mean(axis=1).round(2) + 0)
print()
print("Average standardized rating by movie:")
print(standardized.mean().round(2))

Average standardized rating by rater:
Rater
Alex       0.0
Brianna    0.0
Carlos     0.0
Dana       0.0
Evan       0.0
Fatima     0.0
dtype: Float64

Average standardized rating by movie:
Sinners                   0.57
Superman                  0.16
Jurassic World Rebirth   -1.29
F1                        0.34
Weapons                   0.45
Zootopia 2                0.02
dtype: Float64


Every rater's standardized average is 0, since each score is measured against that
rater's own average. The movie averages give the same ranking as the normalized ones.
Sinners comes out on top, and Jurassic World Rebirth comes last with every method.
Standardizing is based on how spread out each person's ratings are, not just their
highest and lowest score, so one unusual rating has less effect on the result.